<a href="https://colab.research.google.com/github/wnstj1126-debug/-/blob/main/DataInventory_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DeepBind 예지보전 — 1단계: 데이터 인벤토리
**목적**: IMS / FEMTO / CWRU 데이터셋의 실제 구조를 있는 그대로 파악  
**원칙**: 이 단계에서는 리샘플링 / 슬라이싱 / 학습 없음. 순수 파악만.  
**근거**: CLAUDE.md §6(데이터 정책), §10(OPEN 이슈), §11-1(다음 작업)

| 파라미터 | 값 |
|:---|:---|
| 작성일 | 2026-07-24 |
| 최우선 확인 | native sample rate + Nyquist (일괄 업샘플링 금지 원칙) |
| 산출물 | 채널구성·fs·축 물리의미 요약표 |

### 실행 순서
셀A(경로 설정) → 셀B(공통 유틸) → 셀B-2(CWRU manifest + Paderborn npz) → 셀C(폴더 스캔) → **셀C-2(전처리 자산 shape 확인)** → 셀D(스펙 대조) → 셀E(요약표)


In [14]:
# ============================================================
# 셀 A — 경로 설정 + 공식 스펙 참조값
# 드라이브 구조 기준 (2026-07-24 확정)
# MyDrive/Colab Notebooks/ 아래 번호 폴더 구조:
#   1.CWRU_original_file_mat_20260504       ← CWRU mat 원본
#   1-1.CWRU_csv_analysis_toolkit           ← CWRU manifest/toolkit
#   1-2.CWRU_raw_4folder(mat)_20260504      ← CWRU raw mat (4폴더)
#   2.Paderborn_raw_mat                     ← Paderborn mat 원본
#   2-1.Paderborn_csv_file_CPU전용_kaggle   ← Paderborn CSV (kaggle)
#   2-2.paderborn_w512샘플_기준미달.npz     ← Paderborn npz 512샘플 (기준미달)
#   2-3.paderborn_w4096.npz                 ← Paderborn npz 4096샘플
#   2-4.Paderborn_iis3dwb_cls3              ← Paderborn cls3 전처리
#   3.RunToFailure_Raw_Data                 ← IMS + FEMTO
# ============================================================

import os, glob
import numpy as np
import pandas as pd
from pathlib import Path

# Google Drive 마운트 (Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE = '/content/drive/MyDrive'
except Exception:
    DRIVE = '/content/drive/MyDrive'
    print('(Colab 아님 또는 이미 마운트됨)')

NB = DRIVE + '/Colab Notebooks'

# ── 경로 확정 (2026-07-24 드라이브 구조 기준) ─────────────
PATHS = {
    # CWRU
    'CWRU_mat':      NB + '/1.CWRU_original_file_mat_20260504',
    'CWRU_toolkit':  NB + '/1-1.CWRU_csv_analysis_toolkit',
    # Paderborn
    'PDB_mat':       NB + '/2.Paderborn_raw_mat',
    'PDB_csv':       NB + '/2-1.Paderborn_csv_file_CPU전용_kaggle',
    'PDB_npz_512':   NB + '/2-2.paderborn_w512샘플_기준미달.npz',
    'PDB_npz_4096':  NB + '/2-3.paderborn_w4096.npz',
    'PDB_cls3':      NB + '/2-4.Paderborn_iis3dwb_cls3',
    # Run-to-Failure (IMS + FEMTO)
    'RTF_root':      NB + '/3.RunToFailure_Raw_Data',
}

# scan_dataset에 쓰는 DATASET_ROOTS (셀C 호환)
DATASET_ROOTS = {
    'CWRU':  PATHS['CWRU_mat'],
    'FEMTO': PATHS['RTF_root'] + '/PRONOSTIA_FEMTO_Bearing',
    'IMS':   PATHS['RTF_root'] + '/IMS_bearing',
}

# ── 문헌 스펙 (실측 대조용 참조값, 맹신 금지) ─────────────
KNOWN_SPECS = {
    'FEMTO': {
        'fs_hz': 25600,
        'snapshot_sec': 0.1,
        'interval_sec': 10,
        'note': 'acc 파일당 0.1초(2560샘플) 스냅샷, 파일간 10초 간격, H/V 2축',
    },
    'IMS': {
        'fs_hz': 20480,
        'note': '20.48kHz 연속녹음, run-to-failure, 4베어링 다채널',
    },
    'CWRU': {
        'fs_hz_options': [12000, 48000],
        'note': '12k 또는 48k, DE/FE/BA 채널, .mat 형식',
    },
}

# ── 경로 존재 여부 일괄 확인 ──────────────────────────────
print('='*60)
print('[경로 확인]')
print('='*60)
for name, path in PATHS.items():
    exists = Path(path).exists()
    tag = '✅' if exists else '❌'
    print(f'  {tag} [{name}]')
    print(f'     {path}')

print()
print('[DATASET_ROOTS]')
for name, path in DATASET_ROOTS.items():
    exists = Path(path).exists()
    print(f'  {"✅" if exists else "❌ 수정필요"} {name}: {path}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[경로 확인]
  ✅ [CWRU_mat]
     /content/drive/MyDrive/Colab Notebooks/1.CWRU_original_file_mat_20260504
  ✅ [CWRU_toolkit]
     /content/drive/MyDrive/Colab Notebooks/1-1.CWRU_csv_analysis_toolkit
  ✅ [PDB_mat]
     /content/drive/MyDrive/Colab Notebooks/2.Paderborn_raw_mat
  ✅ [PDB_csv]
     /content/drive/MyDrive/Colab Notebooks/2-1.Paderborn_csv_file_CPU전용_kaggle
  ✅ [PDB_npz_512]
     /content/drive/MyDrive/Colab Notebooks/2-2.paderborn_w512샘플_기준미달.npz
  ✅ [PDB_npz_4096]
     /content/drive/MyDrive/Colab Notebooks/2-3.paderborn_w4096.npz
  ✅ [PDB_cls3]
     /content/drive/MyDrive/Colab Notebooks/2-4.Paderborn_iis3dwb_cls3
  ✅ [RTF_root]
     /content/drive/MyDrive/Colab Notebooks/3.RunToFailure_Raw_Data

[DATASET_ROOTS]
  ✅ CWRU: /content/drive/MyDrive/Colab Notebooks/1.CWRU_original_file_mat_20260504
  ✅ FEMTO: /content/drive/MyDrive/Colab Notebooks/3.RunTo

In [15]:
# ============================================================
# 셀 B — 공통 유틸: 파일 형식별 peek 함수
# (전체 로드 최소화 — shape 파악이 목적)
# ============================================================

def peek_csv(path, max_rows=5000):
    """CSV/txt의 shape와 샘플값 파악. 대용량 대비 일부만 읽음."""
    try:
        df = None
        for sep in [',', r'\s+', ';', '\t']:
            try:
                df = pd.read_csv(path, sep=sep, header=None,
                                 nrows=max_rows, engine='python')
                if df.shape[1] > 1:
                    break
            except Exception:
                continue
        if df is None:
            return {'ok': False, 'err': '구분자 자동감지 실패'}
        with open(path, 'r', errors='ignore') as fh:
            n_rows = sum(1 for _ in fh)
        return {
            'ok': True,
            'n_cols': df.shape[1],
            'n_rows_total': n_rows,
            'sample_head': df.head(2).values.tolist(),
            'err': None,
        }
    except Exception as e:
        return {'ok': False, 'n_cols': None, 'n_rows_total': None,
                'sample_head': None, 'err': str(e)}


def peek_mat(path):
    """MATLAB .mat 파일의 변수 키와 shape 파악."""
    try:
        from scipy.io import loadmat
        m = loadmat(path)
        keys = {k: (np.array(v).shape if hasattr(v, 'shape') else type(v).__name__)
                for k, v in m.items() if not k.startswith('__')}
        return {'ok': True, 'keys': keys, 'err': None}
    except Exception as e:
        return {'ok': False, 'keys': None, 'err': str(e)}


def peek_npy(path):
    try:
        arr = np.load(path, mmap_mode='r')
        return {'ok': True, 'shape': arr.shape, 'dtype': str(arr.dtype), 'err': None}
    except Exception as e:
        return {'ok': False, 'shape': None, 'err': str(e)}


print('유틸 함수 정의 완료 (peek_csv / peek_mat / peek_npy)')

유틸 함수 정의 완료 (peek_csv / peek_mat / peek_npy)


In [16]:
# ============================================================
# 셀 B-2 — CWRU manifest + Paderborn npz 규격 확인
# 경로: 셀A의 PATHS 변수 사용 (경로 중복 정의 없음)
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

# ── (1) CWRU manifest 분석 ────────────────────────────────
print('='*60)
print('[CWRU manifest 분석]')
print('='*60)

# manifest 위치: 1-1.CWRU_csv_analysis_toolkit 안에 있음
CWRU_MANIFEST = PATHS['CWRU_toolkit'] + '/cwru_download_manifest.csv'

try:
    m = pd.read_csv(CWRU_MANIFEST)
    print(f'총 {len(m)}개 파일 정리됨')
    print('[컬럼 전체]', list(m.columns), '\n')

    # 고장유형별 집계
    for col in ['fault_type', 'fault_location', 'fault_size', 'fault_diameter_inch']:
        if col in m.columns:
            print(f'[{col} 분포]')
            print(m[col].value_counts().sort_index().to_string(), '\n')

    # RPM 분포 — O-A(1800RPM 가정) 근거
    for col in ['rpm_approx', 'rpm', 'RPM', 'speed']:
        if col in m.columns:
            print(f'[{col} 분포]  ← O-A: PoC 1호 모터 RPM 가정 근거')
            print(m[col].value_counts().sort_index().to_string(), '\n')
            break

    # 부하별
    for col in ['load_HP', 'load', 'HP']:
        if col in m.columns:
            print(f'[{col}(부하) 분포]')
            print(m[col].value_counts().sort_index().to_string(), '\n')
            break

    # fs: 컬럼 or 파일명 기반
    fs_found = False
    for col in ['fs', 'sample_rate', 'fs_hz', 'sampling_rate']:
        if col in m.columns:
            print(f'[{col} 분포]  ← 계보 A: 12k/48k 구분 근거')
            print(m[col].value_counts().sort_index().to_string(), '\n')
            fs_found = True; break
    if not fs_found:
        fn_col = next((c for c in ['filename','file','name'] if c in m.columns), None)
        if fn_col:
            m['_fs'] = m[fn_col].str.extract(r'(12k|48k)', expand=False)
            if m['_fs'].notna().any():
                print('[파일명 기반 fs 분포]  ← 12k vs 48k 자동 분류')
                print(m['_fs'].value_counts().to_string(), '\n')
    print('✅ CWRU manifest 분석 완료')

except FileNotFoundError:
    # toolkit 폴더 안 파일 목록 출력으로 대체
    tk = Path(PATHS['CWRU_toolkit'])
    if tk.exists():
        files = list(tk.rglob('*.csv'))
        print(f'⚠️ manifest.csv 없음. toolkit 폴더 내 CSV:')
        for f in files[:10]:
            print(f'   {f.name}')
    else:
        print(f'⚠️ CWRU_toolkit 폴더 없음: {PATHS["CWRU_toolkit"]}')
except Exception as e:
    print(f'⚠️ 오류: {e}')


# ── (2) Paderborn npz 규격 확인 ───────────────────────────
print('\n' + '='*60)
print('[Paderborn npz 규격 확인]')
print('='*60)

npz_targets = {
    'PDB_npz_512  (기준미달 archive)': PATHS['PDB_npz_512'],
    'PDB_npz_4096 (§4 주력 후보)':    PATHS['PDB_npz_4096'],
}

for label, npz_path in npz_targets.items():
    print(f'\n📦 {label}')
    print(f'   경로: {npz_path}')
    p = Path(npz_path)

    # 파일 직접 지정인 경우
    candidates = [p] if p.suffix == '.npz' else list(p.rglob('*.npz'))[:5]

    if not candidates or not any(c.exists() for c in candidates):
        print(f'   ❌ 파일 없음')
        continue

    for npz_file in candidates:
        if not npz_file.exists():
            continue
        print(f'   파일: {npz_file.name}  ({npz_file.stat().st_size//1024//1024:.1f}MB)')
        try:
            data = np.load(str(npz_file), allow_pickle=True)
            for key in data.files:
                arr = data[key]
                shape = getattr(arr, 'shape', 'scalar')
                dtype = getattr(arr, 'dtype', '?')
                print(f'   - {key}: shape={shape}, dtype={dtype}')
                if hasattr(shape, '__len__') and len(shape) >= 2:
                    w = shape[-1]
                    tag = {512: '⚠️ 기준미달(계보A 검증값)', 2048: '✅ §4 비교군', 4096: '✅ §4 주력'}.get(w, f'?({w}샘플)')
                    print(f'     → 윈도우={w}샘플: {tag}')
        except Exception as e:
            print(f'   ⚠️ 로드 실패: {e}')

# PDB_csv 구조도 간단히
print('\n' + '-'*40)
print('[Paderborn CSV (2-1 폴더) 구조]')
pdb_csv_dir = Path(PATHS['PDB_csv'])
if pdb_csv_dir.exists():
    csv_files = list(pdb_csv_dir.rglob('*.csv'))
    print(f'  CSV 파일 수: {len(csv_files)}')
    for f in csv_files[:5]:
        print(f'  - {f.name}  ({f.stat().st_size//1024}KB)')
else:
    print(f'  ❌ 경로 없음: {PATHS["PDB_csv"]}')

print('\n' + '='*60)
print('[B-2 확인 포인트 요약]')
print('='*60)
print('  CWRU:')
print('    □ manifest fs(12k/48k) 컬럼 or 파일명 분류 → 계보A fs=12,000Hz 확정')
print('    □ RPM 분포 → O-A(1800RPM 가정) 근거 여부')
print('  Paderborn npz:')
print('    □ 2-2(512샘플): 기준미달 archive — 학습/비교 사용 금지')
print('    □ 2-3(4096샘플): §4 주력 후보 규격 확인')
print('    □ CLAUDE.md §6: Paderborn = 물리 검증 archive 전용 (학습용 아님)')

[CWRU manifest 분석]
총 161개 파일 정리됨
[컬럼 전체] ['category', 'orig_label', 'fault_type', 'fault_diameter_inch', 'load_HP', 'rpm_approx', 'or_position', 'src_file_num', 'src_url', 'new_filename', 'aidrive_path'] 

[fault_type 분포]
fault_type
Ball          40
Inner Race    40
Normal         4
Outer Race    77 

[fault_diameter_inch 분포]
fault_diameter_inch
0.007    60
0.014    37
0.021    52
0.028     8 

[rpm_approx 분포]  ← O-A: PoC 1호 모터 RPM 가정 근거
rpm_approx
1730    40
1750    40
1772    40
1797    41 

[load_HP(부하) 분포]
load_HP
0    41
1    40
2    40
3    40 

✅ CWRU manifest 분석 완료

[Paderborn npz 규격 확인]

📦 PDB_npz_512  (기준미달 archive)
   경로: /content/drive/MyDrive/Colab Notebooks/2-2.paderborn_w512샘플_기준미달.npz
   파일: 2-2.paderborn_w512샘플_기준미달.npz  (0.0MB)
   ⚠️ 로드 실패: [Errno 21] Is a directory: '/content/drive/MyDrive/Colab Notebooks/2-2.paderborn_w512샘플_기준미달.npz'

📦 PDB_npz_4096 (§4 주력 후보)
   경로: /content/drive/MyDrive/Colab Notebooks/2-3.paderborn_w4096.npz
   파일: 2-3.paderborn_w4096.npz  (0.0

In [17]:
# ============================================================
# 셀 C — 파일 인벤토리 스캔
# 각 데이터셋 폴더에 뭐가 있는지 확장자별 집계 + 대표 파일 구조 열람
# ============================================================

def scan_dataset(name, root):
    root = Path(root)
    print(f"\n{'='*60}")
    print(f"[{name}] 스캔 중: {root}")
    print('='*60)

    if not root.exists():
        print(f'  ⚠️ 경로 없음 → DATASET_ROOTS["{name}"] 수정 필요')
        return {'name': name, 'exists': False, 'files': []}

    ext_count = {}
    all_files = []
    for p in root.rglob('*'):
        if p.is_file():
            ext = p.suffix.lower()
            ext_count[ext] = ext_count.get(ext, 0) + 1
            all_files.append(p)

    print(f'  총 파일 수: {len(all_files)}')
    print(f'  확장자별:  {ext_count}')

    # 확장자당 첫 파일만 샘플로 열람
    samples = []
    seen_ext = set()
    for p in sorted(all_files):
        ext = p.suffix.lower()
        if ext in seen_ext:
            continue
        seen_ext.add(ext)

        info = {
            'path': str(p.relative_to(root)),
            'ext': ext,
            'size_kb': round(p.stat().st_size / 1024, 1),
        }
        if ext in ['.csv', '.txt']:
            info.update(peek_csv(str(p)))
        elif ext == '.mat':
            info.update(peek_mat(str(p)))
        elif ext == '.npy':
            info.update(peek_npy(str(p)))
        samples.append(info)

    print('\n  [대표 파일 구조]')
    for s in samples:
        print(f"  - {s['path']} ({s['size_kb']}KB)")
        if s.get('n_cols') is not None:
            print(f"    → {s['n_rows_total']}행 × {s['n_cols']}열")
            print(f"    → head[0]: {s.get('sample_head', [[]])[0]}")
        elif s.get('keys') is not None:
            print(f"    → .mat keys: {s['keys']}")
        elif s.get('shape') is not None:
            print(f"    → npy shape: {s['shape']}, dtype: {s.get('dtype')}")
        if s.get('err'):
            print(f"    ⚠️ 오류: {s['err']}")

    return {
        'name': name,
        'exists': True,
        'n_files': len(all_files),
        'ext_count': ext_count,
        'samples': samples,
        'all_files': [str(p) for p in all_files],
    }


# 실행
inventory = {}
for name, root in DATASET_ROOTS.items():
    inventory[name] = scan_dataset(name, root)


[CWRU] 스캔 중: /content/drive/MyDrive/Colab Notebooks/1.CWRU_original_file_mat_20260504
  총 파일 수: 161
  확장자별:  {'.mat': 161}

  [대표 파일 구조]
  - B007_0_load0HP_1797rpm_src118.mat (2873.2KB)
    → .mat keys: {'X118_DE_time': (122571, 1), 'X118_FE_time': (122571, 1), 'X118_BA_time': (122571, 1), 'X118RPM': (1, 1)}

[FEMTO] 스캔 중: /content/drive/MyDrive/Colab Notebooks/3.RunToFailure_Raw_Data/PRONOSTIA_FEMTO_Bearing
  총 파일 수: 43369
  확장자별:  {'.csv': 43369}

  [대표 파일 구조]
  - Test(Test)_set/Bearing1_3/acc_00001.csv (76.6KB)
    → 2560행 × 6열
    → head[0]: [8.0, 33.0, 1.0, 378160.0, 0.092, 0.044]

[IMS] 스캔 중: /content/drive/MyDrive/Colab Notebooks/3.RunToFailure_Raw_Data/IMS_bearing
  총 파일 수: 9465
  확장자별:  {'.pdf': 1, '.57': 3838, '.46': 2018, '.20': 523, '.17': 40, '.53': 1, '.55': 389, '.39': 985, '.56': 281, '.11': 1, '.24': 82, '.30': 201, '.32': 100, '.13': 153, '.48': 1, '.38': 1, '.44': 421, '.51': 1, '.19': 2, '.58': 172, '.15': 1, '.09': 1, '.07': 68, '.35': 1, '.03': 158, '.21': 2, '.3

In [22]:
# ============================================================
# 셀 C-2 — 핵심 전처리 자산 shape 확인 (새 파일 생성 없음, 확인만)
# 목적: Paderborn 4096 정합성 + CWRU 기추출 특징 + iis3dwb cls3 진단
# 경로: 셀A의 PATHS 변수 사용
# ============================================================

import numpy as np
import pandas as pd
import json
from pathlib import Path

# ── (1) Paderborn 4096 npz — §4 정합성 핵심 ────────────────
print('='*60)
print('[1] Paderborn 4096 npz 규격 확인  (CLAUDE.md §4 주력 후보)')
print('='*60)

pdb4096_path = Path(PATHS['PDB_npz_4096'])

# 직접 파일이면 리스트로, 폴더면 rglob
pdb4096_files = [] # Initialize list for .npz files

if pdb4096_path.is_file() and pdb4096_path.suffix == '.npz':
    pdb4096_files = [pdb4096_path] # 진짜 단일 파일
elif pdb4096_path.is_dir():
    # 폴더 안 탐색
    all_npz_in_dir = list(pdb4096_path.rglob('*.npz'))
    if all_npz_in_dir:
        pdb4096_files = all_npz_in_dir[:5] # Take up to 5 npz files
    else:
        # .npz 없으면 폴더 내 파일 목록 출력 (무엇이 있는지 파악)
        print(f'  ⚠️ {pdb4096_path} 폴더에 .npz 파일이 없습니다. 폴더 내용:')
        dir_contents = list(pdb4096_path.iterdir())
        for f in dir_contents[:5]: # Print first 5 items
            print(f'    - {f.name}{' (디렉토리)' if f.is_dir() else ''}')
else:
    # Path does not exist, or it's not a file/directory of interest
    print(f'  ⚠️ "{pdb4096_path}" 경로가 .npz 파일이거나 유효한 폴더가 아닙니다.')

if not pdb4096_files:
    print(f'  ❌ npz 없음: {pdb4096_path}')
else:
    for npz_p in pdb4096_files:
        print(f'\n📦 {npz_p.name}  ({npz_p.stat().st_size//1024//1024:.1f}MB)')
        try:
            d = np.load(str(npz_p), allow_pickle=True)
            for k in d.files:
                a = d[k]
                shape = getattr(a, 'shape', '?')
                dtype = getattr(a, 'dtype', '?')
                print(f'  {k}: shape={shape}, dtype={dtype}')
                if hasattr(shape, '__len__') and len(shape) >= 2:
                    w = shape[-1]
                    # 윈도우 길이 해석
                    t_iis = w / 26667 * 1000   # IIS3DWB 기준 ms
                    t_pdb = w / 64000 * 1000   # Paderborn 64kHz 기준 ms
                    tag = {
                        512:  '⚠️ 기준미달(계보A 검증값, 0.58회전@1800RPM)',
                        2048: '✅ §4 비교군 (76.8ms @IIS3DWB, 2.3회전)',
                        4096: '✅ §4 주력 (153.6ms @IIS3DWB, 4.6회전)',
                    }.get(w, f'?({w}샘플)')
                    print(f'    → 윈도우={w}샘플: {tag}')
                    print(f'       @IIS3DWB 26.7kHz = {t_iis:.1f}ms'
                          f' / @Paderborn 64kHz = {t_pdb:.1f}ms')
                    if len(shape) >= 1:
                        print(f'       샘플(윈도우) 수: {shape[0]}개')
        except Exception as e:
            print(f'  ⚠️ 로드 실패: {e}')

# ── (2) CWRU 기추출 특징 확인 ─────────────────────────────
print('\n' + '='*60)
print('[2] CWRU 기추출 특징 (cwru_features.csv)')
print('    계보B M0(9특징 Mahalanobis)와 특징 구성 비교')
print('='*60)

cwru_feat_path = PATHS['CWRU_toolkit'] + '/cwru_features.csv'
try:
    cf = pd.read_csv(cwru_feat_path)
    print(f'  shape: {cf.shape}  → 특징 {cf.shape[1]}개 컬럼')
    print(f'  컬럼: {list(cf.columns)}')
    n_feat = cf.shape[1]
    if n_feat == 9:
        print(f'  → ✅ M0와 동일 9특징 구성')
    elif n_feat > 9:
        print(f'  → ℹ️  M0보다 {n_feat-9}개 더 많음 — 어떤 특징 추가됐는지 확인')
    else:
        print(f'  → ⚠️ M0(9특징)보다 적음 ({n_feat}개) — 구성 차이 확인 필요')
    print()
    print(cf.head(3).to_string())
except FileNotFoundError:
    # cwru_features.csv 없으면 toolkit 폴더 내 csv 목록으로 대체
    tk = Path(PATHS['CWRU_toolkit'])
    if tk.exists():
        csvs = list(tk.rglob('*.csv'))
        print(f'  ⚠️ cwru_features.csv 없음. toolkit 내 CSV ({len(csvs)}개):')
        for f in csvs[:8]:
            print(f'    {f.name}  ({f.stat().st_size//1024}KB)')
    else:
        print(f'  ❌ CWRU_toolkit 경로 없음: {PATHS["CWRU_toolkit"]}')
except Exception as e:
    print(f'  ⚠️ 오류: {e}')

# ── (3) Paderborn iis3dwb cls3 진단 ───────────────────────
print('\n' + '='*60)
print('[3] Paderborn iis3dwb cls3 (2-4 폴더)')
print('    AI_KimBanjang_Paderborn_IIS3DWB_v4 실패 관련 자산')
print('='*60)

cls3_dir = Path(PATHS['PDB_cls3'])
if not cls3_dir.exists():
    print(f'  ❌ 폴더 없음: {cls3_dir}')
else:
    # 파일 목록
    all_f = list(cls3_dir.rglob('*'))
    files = [f for f in all_f if f.is_file()]
    print(f'  파일 수: {len(files)}개')
    ext_count = {}
    for f in files:
        ext_count[f.suffix] = ext_count.get(f.suffix, 0) + 1
    print(f'  확장자별: {ext_count}')

    # manifest.json 확인
    mj = cls3_dir / 'manifest.json'
    if mj.exists():
        with open(str(mj)) as fh:
            man = json.load(fh)
        if isinstance(man, dict):
            print(f'\n  manifest.json keys: {list(man.keys())[:10]}')
            print(f'  (크기: {mj.stat().st_size//1024}KB)')
    else:
        print('  manifest.json 없음')

    # npy/npz 샘플 shape 출력
    for ext, label in [('.npy', 'npy'), ('.npz', 'npz')]:
        samples = [f for f in files if f.suffix == ext][:2]
        for sp in samples:
            try:
                if ext == '.npy':
                    a = np.load(str(sp))
                    print(f'\n  📦 {sp.name}: shape={a.shape}, dtype={a.dtype}')
                    w = a.shape[-1] if a.ndim > 1 else len(a)
                    print(f'     윈도우 후보: {w}샘플')
                else:
                    d = np.load(str(sp), allow_pickle=True)
                    print(f'\n  📦 {sp.name}:')
                    for k in d.files:
                        a = d[k]
                        print(f'     {k}: shape={getattr(a,"shape","?")}, dtype={getattr(a,"dtype","?")}')
            except Exception as e:
                print(f'  ⚠️ {sp.name} 로드 실패: {e}')

print('\n' + '='*60)
print('[C-2 확인 포인트 요약]')
print('='*60)
print('  □ Paderborn 4096 npz: shape[-1]=4096 이면 §4 주력 규격 일치')
print('  □ CWRU 특징 CSV: 9특징이면 계보B M0와 동일 구성')
print('  □ cls3 자산: v4 실패 원인(raw vs CSV 입력 불일치) 확인용 archive')
print('  ★ 이 셀은 확인만 — 변환/학습/저장 없음')


[1] Paderborn 4096 npz 규격 확인  (CLAUDE.md §4 주력 후보)

📦 tmp_K001.npz  (33.0MB)
  X: shape=(4964, 4096), dtype=float32
    → 윈도우=4096샘플: ✅ §4 주력 (153.6ms @IIS3DWB, 4.6회전)
       @IIS3DWB 26.7kHz = 153.6ms / @Paderborn 64kHz = 64.0ms
       샘플(윈도우) 수: 4964개
  y_binary: shape=(4964,), dtype=int32
  y_multi: shape=(4964,), dtype=int32
  bearing_codes: shape=(4964,), dtype=<U4

📦 tmp_K002.npz  (30.0MB)
  X: shape=(4975, 4096), dtype=float32
    → 윈도우=4096샘플: ✅ §4 주력 (153.6ms @IIS3DWB, 4.6회전)
       @IIS3DWB 26.7kHz = 153.6ms / @Paderborn 64kHz = 64.0ms
       샘플(윈도우) 수: 4975개
  y_binary: shape=(4975,), dtype=int32
  y_multi: shape=(4975,), dtype=int32
  bearing_codes: shape=(4975,), dtype=<U4

📦 tmp_K003.npz  (32.0MB)
  X: shape=(4974, 4096), dtype=float32
    → 윈도우=4096샘플: ✅ §4 주력 (153.6ms @IIS3DWB, 4.6회전)
       @IIS3DWB 26.7kHz = 153.6ms / @Paderborn 64kHz = 64.0ms
       샘플(윈도우) 수: 4974개
  y_binary: shape=(4974,), dtype=int32
  y_multi: shape=(4974,), dtype=int32
  bearing_codes: shape=(4

In [23]:
# ============================================================
# 셀 D — 실측 검증: 공식 스펙 vs 실제 파일 대조
# CLAUDE.md §6 — Nyquist/유효대역 확정을 위한 핵심 단계
# ============================================================

def verify_specs(name, scan_result):
    print(f"\n{'='*60}")
    print(f"[공식 스펙 vs 실측 대조 — {name}]")
    print('='*60)
    spec = KNOWN_SPECS.get(name, {})
    print(f"  문헌값: {spec.get('note', '?')}")

    if not scan_result.get('exists'):
        print('  ⚠️ 데이터 없어 대조 불가')
        return

    # ── FEMTO ──────────────────────────────────────────────
    if name == 'FEMTO':
        for s in scan_result['samples']:
            if s.get('n_rows_total'):
                n = s['n_rows_total']
                fs = spec.get('fs_hz', 25600)
                t_ms = n / fs * 1000
                print(f"  파일당 {n}행  → 실측 윈도우 시간 = {t_ms:.1f}ms"
                      f"  (문헌 100ms {'✅' if abs(t_ms-100)<5 else '⚠️ 불일치'})")
                n_col = s.get('n_cols', '?')
                print(f"  컬럼 {n_col}개  (FEMTO 표준: 6열 = hour/min/sec/micro/H축/V축)")
                print(f"  Nyquist = {fs//2:,}Hz  (유효대역 상한)")
                if s.get('sample_head'):
                    print(f"  head[0] = {s['sample_head'][0]}")
                print()
                print('  ★ 확인 포인트:')
                print('    - col4(인덱스 4)가 H축, col5가 V축인지 눈으로 확인')
                print('    - OnlineSim_FEMTO_v3.ipynb의 COL_H=4 재확인')

    # ── IMS ────────────────────────────────────────────────
    elif name == 'IMS':
        for s in scan_result['samples']:
            if s.get('n_cols'):
                fs = spec.get('fs_hz', 20480)
                n = s['n_rows_total']
                t_sec = n / fs
                print(f"  채널(열) {s['n_cols']}개, {n}행")
                print(f"  파일당 시간 = {t_sec:.3f}초  ({t_sec*1000:.1f}ms)")
                print(f"  Nyquist = {fs//2:,}Hz")
                print()
                print('  ★ 확인 포인트:')
                print('    - 열 개수: 4베어링×1축=4열 / 4베어링×2축=8열?')
                print('    - 어느 열이 어느 베어링인지 문서(IMS readme) 대조 필수')
                print('    ⚠️ 26,667Hz로 업샘플 금지 (CLAUDE.md §6)')

    # ── CWRU ───────────────────────────────────────────────
    elif name == 'CWRU':
        for s in scan_result['samples']:
            if s.get('keys'):
                print(f"  .mat 변수: {list(s['keys'].keys())}")
                print(f"  변수별 shape: {s['keys']}")
                print()
                print('  ★ 확인 포인트:')
                print('    - 파일명에 "12k" 포함 → fs=12,000Hz, Nyquist=6,000Hz')
                print('    - 파일명에 "48k" 포함 → fs=48,000Hz, Nyquist=24,000Hz')
                print('    - 변수명: *_DE_time(Drive End), *_FE_time(Fan End),')
                print('              *_BA_time(Ball), RPM')
                print('    - 계보 A 노트북 확인: WINDOW=512, STEP=512, fs=12,000Hz')
                print('      → 윈도우 시간 = 512/12000 = 42.7ms (non-overlap)')


for name, result in inventory.items():
    verify_specs(name, result)


[공식 스펙 vs 실측 대조 — CWRU]
  문헌값: 12k 또는 48k, DE/FE/BA 채널, .mat 형식
  .mat 변수: ['X118_DE_time', 'X118_FE_time', 'X118_BA_time', 'X118RPM']
  변수별 shape: {'X118_DE_time': (122571, 1), 'X118_FE_time': (122571, 1), 'X118_BA_time': (122571, 1), 'X118RPM': (1, 1)}

  ★ 확인 포인트:
    - 파일명에 "12k" 포함 → fs=12,000Hz, Nyquist=6,000Hz
    - 파일명에 "48k" 포함 → fs=48,000Hz, Nyquist=24,000Hz
    - 변수명: *_DE_time(Drive End), *_FE_time(Fan End),
              *_BA_time(Ball), RPM
    - 계보 A 노트북 확인: WINDOW=512, STEP=512, fs=12,000Hz
      → 윈도우 시간 = 512/12000 = 42.7ms (non-overlap)

[공식 스펙 vs 실측 대조 — FEMTO]
  문헌값: acc 파일당 0.1초(2560샘플) 스냅샷, 파일간 10초 간격, H/V 2축
  파일당 2560행  → 실측 윈도우 시간 = 100.0ms  (문헌 100ms ✅)
  컬럼 6개  (FEMTO 표준: 6열 = hour/min/sec/micro/H축/V축)
  Nyquist = 12,800Hz  (유효대역 상한)
  head[0] = [8.0, 33.0, 1.0, 378160.0, 0.092, 0.044]

  ★ 확인 포인트:
    - col4(인덱스 4)가 H축, col5가 V축인지 눈으로 확인
    - OnlineSim_FEMTO_v3.ipynb의 COL_H=4 재확인

[공식 스펙 vs 실측 대조 — IMS]
  문헌값: 20.48kHz 연속녹음, run-to-failure, 4베어링 다채널


In [20]:
# ============================================================
# 셀 E — 최종 요약표 (CLAUDE.md §11-1 산출물)
# ============================================================

print('\n' + '='*65)
print('  [최종 인벤토리 요약표]')
print('='*65)

rows = []
for name, r in inventory.items():
    spec = KNOWN_SPECS.get(name, {})
    fs_raw = spec.get('fs_hz', spec.get('fs_hz_options', '?'))

    if not r.get('exists'):
        rows.append({
            '데이터셋': name, '상태': '❌경로없음',
            'fs_Hz': '?', 'Nyquist_Hz': '?',
            '파일수': 0, '채널(열)': '?', '비고': '경로 수정 필요',
        })
        continue

    fs = fs_raw if isinstance(fs_raw, int) else '12k/48k 혼재'
    nyq = (fs_raw // 2) if isinstance(fs_raw, int) else '확인필요'

    n_ch = '?'
    fmt = '?'
    for s in r['samples']:
        if s.get('n_cols'):
            n_ch = s['n_cols']; fmt = s['ext']; break
        if s.get('keys'):
            n_ch = str(len(s['keys'])) + ' vars'; fmt = '.mat'; break

    note_map = {
        'FEMTO': 'H/V 2축 스냅샷, 10초 간격',
        'IMS':   '4베어링 연속 run-to-failure',
        'CWRU':  'DE/FE/BA 채널, 12k or 48k',
    }
    rows.append({
        '데이터셋': name,
        '상태': '✅',
        'fs_Hz': fs,
        'Nyquist_Hz': nyq,
        '파일수': r.get('n_files', 0),
        '채널(열)': n_ch,
        '비고': note_map.get(name, ''),
    })

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))

print()
print('='*65)
print('[사람이 직접 확인해야 할 항목]')
print('  1. FEMTO: col4=H축 / col5=V축 맞는지 head 값으로 눈으로 확인')
print('  2. IMS: 열 개수가 4개(베어링당 1축)인지 8개(2축)인지 확정')
print('     → 어느 열이 어느 베어링인지 IMS readme 대조')
print('  3. CWRU: 파일명별 12k/48k 구분 → 계보 A fs=12,000Hz 확정됨')
print('  4. 세 데이터셋 Nyquist가 다름:')
print('     FEMTO=12,800Hz / IMS=10,240Hz / CWRU-12k=6,000Hz')
print('     → 공통 유효대역 상한 = min(Nyquist) 기준으로 논의 필요')
print('  5. 업샘플링 금지 원칙(CLAUDE.md §6) 재확인 — 이 표가 그 근거')
print('='*65)
print()
print('[다음 단계 — 확인 후 진행]')
print('  M0 재현: FEMTO H축 9특징 Mahalanobis → 5/6 성적표 baseline 고정')
print('  R0/R1/R2 리샘플링 비교 트랙 설계 (fingerprint 점검 포함)')
print('  M1/M2 비교 실험 준비 (FRAME_LEN=4096, DECISION_INTERVAL=0.2s)')


  [최종 인벤토리 요약표]
 데이터셋 상태      fs_Hz Nyquist_Hz   파일수  채널(열)                      비고
 CWRU  ✅ 12k/48k 혼재       확인필요   161 4 vars DE/FE/BA 채널, 12k or 48k
FEMTO  ✅      25600      12800 43369      6      H/V 2축 스냅샷, 10초 간격
  IMS  ✅      20480      10240  9465      ?  4베어링 연속 run-to-failure

[사람이 직접 확인해야 할 항목]
  1. FEMTO: col4=H축 / col5=V축 맞는지 head 값으로 눈으로 확인
  2. IMS: 열 개수가 4개(베어링당 1축)인지 8개(2축)인지 확정
     → 어느 열이 어느 베어링인지 IMS readme 대조
  3. CWRU: 파일명별 12k/48k 구분 → 계보 A fs=12,000Hz 확정됨
  4. 세 데이터셋 Nyquist가 다름:
     FEMTO=12,800Hz / IMS=10,240Hz / CWRU-12k=6,000Hz
     → 공통 유효대역 상한 = min(Nyquist) 기준으로 논의 필요
  5. 업샘플링 금지 원칙(CLAUDE.md §6) 재확인 — 이 표가 그 근거

[다음 단계 — 확인 후 진행]
  M0 재현: FEMTO H축 9특징 Mahalanobis → 5/6 성적표 baseline 고정
  R0/R1/R2 리샘플링 비교 트랙 설계 (fingerprint 점검 포함)
  M1/M2 비교 실험 준비 (FRAME_LEN=4096, DECISION_INTERVAL=0.2s)


## 실행 가이드 & 주의사항

### 경로 수정 (필수)
`DATASET_ROOTS` 딕셔너리의 세 경로를 본인 드라이브 구조에 맞게 수정하세요.  
현재 경로는 이전 실험에서 확인된 경로 기준이며, 실제 위치가 다를 수 있습니다.

### 이 노트북이 하는 일
- 세 데이터셋 폴더를 스캔 → 파일 수·확장자·대표 파일 shape 파악
- 문헌 스펙(fs, 윈도우 시간, 채널 수)과 실제 파일 대조
- native Nyquist 확정 → 일괄 업샘플링 금지 근거 표 출력

### 이 노트북이 하지 않는 일
- 리샘플링 ❌ / 슬라이싱 ❌ / 특징 추출 ❌ / 학습 ❌

### 관련 문서
- `workspace/CLAUDE.md` §6 데이터 정책, §10 OPEN 이슈, §11 다음 작업
- `field_iis3dwb/CLAUDE.md` FEMTO 트랙 종료 기록 (M0 5/6 성적표)
- `field_iis3dwb/OnlineSim_FEMTO_v3.ipynb` M0 실험 노트북